# Wazuh ML - Severity classification

Objectif: entrainer et comparer Logistic Regression, Random Forest et XGBoost pour predire la **severity**.
Notebook complet, avec preprocessing detaille, tuning hyperparametres, evaluation et visualisations.


In [101]:
# Imports et configuration generale
from pathlib import Path
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    balanced_accuracy_score,
    f1_score,
    accuracy_score,
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight
import joblib

try:
    import xgboost as xgb
except Exception:
    xgb = None

SEED = 42
np.random.seed(SEED)

plt.rcParams["figure.figsize"] = (7, 4)


In [102]:
# Chargement du dataset
DATA_PATH = Path("../data/wazuh_logs.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Missing dataset: {DATA_PATH}")

df = pd.read_csv(DATA_PATH, low_memory=False)
print("Shape:", df.shape)
df.head()


Shape: (9125, 11)


,@timestamp,agent.name,agent.id,rule.id,rule.description,rule.level,rule.mitre.id,location,data.win.eventdata.commandLine,raw,severity
0,2025-12-27T14:10:22.179Z,wazuh-vm,0,510,Host-based anomaly detection event (rootcheck).,7,NaN,rootcheck,NaN,"{'agent': {'name': 'wazuh-vm', 'id': '000'}, '...",high
1,2025-12-27T14:10:22.211Z,wazuh-vm,0,510,Host-based anomaly detection event (rootcheck).,7,NaN,rootcheck,NaN,"{'agent': {'name': 'wazuh-vm', 'id': '000'}, '...",high
2,2025-12-27T14:10:29.858Z,wazuh-vm,0,502,Wazuh server started.,3,NaN,wazuh-monitord,NaN,"{'agent': {'name': 'wazuh-vm', 'id': '000'}, '...",low
3,2025-12-27T14:10:31.706Z,wazuh-vm,0,2901,New dpkg (Debian Package) requested to install.,3,NaN,/var/log/dpkg.log,NaN,"{'agent': {'name': 'wazuh-vm', 'id': '000'}, '...",low
4,2025-12-27T14:10:35.717Z,wazuh-vm,0,2904,Dpkg (Debian Package) half configured.,7,NaN,/var/log/dpkg.log,NaN,"{'agent': {'name': 'wazuh-vm', 'id': '000'}, '...",high


In [103]:
# Apercu et types de colonnes
print(df.dtypes)
print("Missing rate top 10:\n", df.isna().mean().sort_values(ascending=False).head(10))


@timestamp                        object
agent.name                        object
agent.id                           int64
rule.id                            int64
rule.description                  object
rule.level                         int64
rule.mitre.id                     object
location                          object
data.win.eventdata.commandLine    object
raw                               object
severity                          object
dtype: object
Missing rate top 10:
 data.win.eventdata.commandLine    0.895123
rule.mitre.id                     0.686795
@timestamp                        0.000000
agent.id                          0.000000
agent.name                        0.000000
rule.description                  0.000000
rule.id                           0.000000
rule.level                        0.000000
location                          0.000000
raw                               0.000000
dtype: float64


In [104]:
# Nettoyage et normalisation du label severity
raw_severity = df.get("severity", pd.Series([None] * len(df)))
sev = raw_severity.astype(str).str.lower().str.strip()

mapping = {
    "low": "low",
    "medium": "medium",
    "med": "medium",
    "high": "high",
    "critical": "critical",
    "crit": "critical",
}

sev = sev.map(mapping)

valid = {"low", "medium", "high", "critical"}
mask_valid = sev.isin(valid)

if (~mask_valid).any():
    bad_count = int((~mask_valid).sum())
    print(f"Dropping {bad_count} rows with invalid severity.")

df = df.loc[mask_valid].copy()
df["severity"] = sev[mask_valid]

print("Severity distribution:\n", df["severity"].value_counts())

# Encodage label pour compatibilite modele
label_encoder = LabelEncoder()
df["severity_encoded"] = label_encoder.fit_transform(df["severity"])
class_labels = list(label_encoder.classes_)
print("Label mapping:", dict(zip(class_labels, range(len(class_labels)))))


Severity distribution:
 severity
low         5840
medium      1473
high        1203
critical     609
Name: count, dtype: int64
Label mapping: {'critical': 0, 'high': 1, 'low': 2, 'medium': 3}


In [105]:
# Nettoyage NA (prepare les imputers, pas encore fit)
text_imputer = SimpleImputer(strategy="constant", fill_value="")
cat_imputer = SimpleImputer(strategy="most_frequent")
num_imputer = SimpleImputer(strategy="median")


In [106]:
# Feature engineering timestamp
if "@timestamp" in df.columns:
    ts = pd.to_datetime(df["@timestamp"], errors="coerce")
    df["hour"] = ts.dt.hour
    df["dayofweek"] = ts.dt.dayofweek
    df["month"] = ts.dt.month
    df = df.drop(columns=["@timestamp"])
    print("Timestamp features added: hour, dayofweek, month")
else:
    df["hour"] = np.nan
    df["dayofweek"] = np.nan
    df["month"] = np.nan
    print("@timestamp not found, using NaN for time features")


Timestamp features added: hour, dayofweek, month


In [107]:
# Selection des features et nettoyage texte minimal

def clean_text_series(series: pd.Series) -> pd.Series:
    cleaned = series.fillna("").astype(str).str.lower()
    cleaned = cleaned.str.replace(r"[\x00-\x1f]+", " ", regex=True)
    cleaned = cleaned.str.replace(r"\s+", " ", regex=True).str.strip()
    return cleaned

# Candidats texte
text_candidates = [
    "rule.description",
    "data.win.eventdata.commandLine",
    "raw",
    "rule.mitre.id",
]

# Candidats categoriels
cat_candidates = ["agent.name", "location", "agent.id", "rule.id"]

# Numeriques
num_candidates = ["rule.level", "hour", "dayofweek", "month"]

existing_cols = set(df.columns)
text_cols = [c for c in text_candidates if c in existing_cols]
cat_cols = [c for c in cat_candidates if c in existing_cols]
num_cols = [c for c in num_candidates if c in existing_cols]

# Nettoyage texte
for c in text_cols:
    df[c] = clean_text_series(df[c])

# Drop categorical ids if cardinality too high
max_card = 50
kept_cat = []
dropped_cat = []
for c in cat_cols:
    n_unique = df[c].nunique(dropna=True)
    if n_unique <= max_card:
        kept_cat.append(c)
    else:
        dropped_cat.append(c)

cat_cols = kept_cat

print("Text cols:", text_cols)
print("Cat cols (kept):", cat_cols)
print("Cat cols (dropped):", dropped_cat)
print("Num cols:", num_cols)

# Define features and target
feature_cols = text_cols + cat_cols + num_cols
X = df[feature_cols].copy()
y = df["severity_encoded"].copy()

print("Final feature columns:", feature_cols)
print("Class labels:", class_labels)


Text cols: ['rule.description', 'data.win.eventdata.commandLine', 'raw', 'rule.mitre.id']
Cat cols (kept): ['agent.name', 'location', 'agent.id']
Cat cols (dropped): ['rule.id']
Num cols: ['rule.level', 'hour', 'dayofweek', 'month']
Final feature columns: ['rule.description', 'data.win.eventdata.commandLine', 'raw', 'rule.mitre.id', 'agent.name', 'location', 'agent.id', 'rule.level', 'hour', 'dayofweek', 'month']
Class labels: ['critical', 'high', 'low', 'medium']


In [108]:
# Split train/test stratifie
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y,
)

print("Train size:", X_train.shape, "Test size:", X_test.shape)


Train size: (7300, 11) Test size: (1825, 11)


In [109]:
# Preprocessor: ColumnTransformer + pipelines
def combine_text(x):
    if hasattr(x, "fillna"):
        return x.fillna("").astype(str).agg(" ".join, axis=1)
    return pd.DataFrame(x).fillna("").astype(str).agg(" ".join, axis=1)

text_pipeline = Pipeline([
    ("imputer", text_imputer),
    ("to_text", FunctionTransformer(combine_text, validate=False)),
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), max_features=5000)),
])

cat_pipeline = Pipeline([
    ("imputer", cat_imputer),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

num_pipeline_scaled = Pipeline([
    ("imputer", num_imputer),
    ("scaler", StandardScaler(with_mean=False)),
])

num_pipeline_plain = Pipeline([
    ("imputer", num_imputer),
])

transformers_scaled = []
transformers_plain = []

if text_cols:
    transformers_scaled.append(("text", text_pipeline, text_cols))
    transformers_plain.append(("text", text_pipeline, text_cols))
if cat_cols:
    transformers_scaled.append(("cat", cat_pipeline, cat_cols))
    transformers_plain.append(("cat", cat_pipeline, cat_cols))
if num_cols:
    transformers_scaled.append(("num", num_pipeline_scaled, num_cols))
    transformers_plain.append(("num", num_pipeline_plain, num_cols))

preprocess_scaled = ColumnTransformer(transformers_scaled)
preprocess_plain = ColumnTransformer(transformers_plain)

if not transformers_scaled:
    raise ValueError("No usable feature columns found. Check text/cat/num columns.")


In [110]:
# Baseline rapide: Logistic Regression (sans tuning)
logreg_base = Pipeline([
    ("preprocess", preprocess_scaled),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced", solver="liblinear")),
])

logreg_base.fit(X_train, y_train)
print("Baseline Logistic Regression trained.")


c:\Users\Kebiyer\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(


Baseline Logistic Regression trained.


In [111]:
# Logistic Regression + RandomizedSearchCV
logreg = Pipeline([
    ("preprocess", preprocess_scaled),
    ("clf", LogisticRegression(max_iter=3000, class_weight="balanced", solver="liblinear")),
])

logreg_param = {
    "clf__C": [0.05, 0.1, 0.5, 1, 3, 10],
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)

logreg_search = RandomizedSearchCV(
    logreg,
    logreg_param,
    n_iter=6,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    random_state=SEED,
)

logreg_search.fit(X_train, y_train)
print("Best LogisticRegression params:", logreg_search.best_params_)


c:\Users\Kebiyer\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1296: FutureWarning: Using the 'liblinear' solver for multiclass classification is deprecated. An error will be raised in 1.8. Either use another solver which supports the multinomial loss or wrap the estimator in a OneVsRestClassifier to keep applying a one-versus-rest scheme.
  warnings.warn(


Best LogisticRegression params: {'clf__C': 10}


In [112]:
# Random Forest + RandomizedSearchCV
# RandomForest needs dense inputs
rf = Pipeline([
    ("preprocess", preprocess_plain),
    ("to_dense", FunctionTransformer(lambda x: x.toarray() if hasattr(x, "toarray") else x, validate=False)),
    ("clf", RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced_subsample",
        random_state=SEED,
    )),
])

rf_param = {
    "clf__max_depth": [None, 10, 20],
    "clf__min_samples_split": [2, 5, 10],
    "clf__min_samples_leaf": [1, 2, 4],
}

rf_search = RandomizedSearchCV(
    rf,
    rf_param,
    n_iter=8,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
    random_state=SEED,
)

rf_search.fit(X_train, y_train)
print("Best RandomForest params:", rf_search.best_params_)


Best RandomForest params: {'clf__min_samples_split': 2, 'clf__min_samples_leaf': 1, 'clf__max_depth': None}


In [ ]:
# XGBoost + RandomizedSearchCV
if xgb is None:
    print("xgboost is not installed. Skipping XGBoost training.")
    xgb_search = None
else:
    # Compute sample weights for class imbalance
    classes = np.unique(y_train)
    class_weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
    weight_map = {cls: w for cls, w in zip(classes, class_weights)}
    sample_weight = y_train.map(weight_map).values

    xgb_model = xgb.XGBClassifier(
        objective="multi:softprob",
        num_class=len(class_labels),
        random_state=SEED,
        n_estimators=300,
        tree_method="hist",
        eval_metric="mlogloss",
    )

    xgb_pipe = Pipeline([
        ("preprocess", preprocess_plain),
        ("to_dense", FunctionTransformer(lambda x: x.toarray() if hasattr(x, "toarray") else x, validate=False)),
        ("clf", xgb_model),
    ])

    xgb_param = {
        "clf__max_depth": [4, 6, 8],
        "clf__min_child_weight": [1, 5, 10],
        "clf__subsample": [0.8, 1.0],
        "clf__colsample_bytree": [0.8, 1.0],
        "clf__reg_lambda": [1.0, 2.0, 5.0],
        "clf__reg_alpha": [0.0, 0.5, 1.0],
        "clf__learning_rate": [0.05, 0.1, 0.2],
    }

    xgb_search = RandomizedSearchCV(
        xgb_pipe,
        xgb_param,
        n_iter=10,
        scoring="f1_macro",
        cv=cv,
        n_jobs=-1,
        random_state=SEED,
    )

    xgb_search.fit(X_train, y_train, clf__sample_weight=sample_weight)
    print("Best XGBoost params:", xgb_search.best_params_)


In [ ]:
# Optionnel: early stopping XGBoost (activez en mettant True)
USE_XGB_EARLY_STOP = False
xgb_es_pipeline = None

if xgb is not None and USE_XGB_EARLY_STOP:
    X_tr, X_val, y_tr, y_val = train_test_split(
        X_train,
        y_train,
        test_size=0.2,
        random_state=SEED,
        stratify=y_train,
    )

    preprocess_plain.fit(X_tr)
    X_tr_p = preprocess_plain.transform(X_tr)
    X_val_p = preprocess_plain.transform(X_val)
    if hasattr(X_tr_p, "toarray"):
        X_tr_p = X_tr_p.toarray()
        X_val_p = X_val_p.toarray()

    xgb_es = xgb.XGBClassifier(
        objective="multi:softprob",
        num_class=len(class_labels),
        random_state=SEED,
        n_estimators=1000,
        learning_rate=0.05,
        max_depth=6,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_lambda=2.0,
        reg_alpha=0.0,
        tree_method="hist",
        eval_metric="mlogloss",
    )

    xgb_es.fit(
        X_tr_p,
        y_tr,
        eval_set=[(X_val_p, y_val)],
        early_stopping_rounds=30,
        verbose=False,
    )

    class PreprocessedModel:
        def __init__(self, preprocessor, model):
            self.preprocessor = preprocessor
            self.model = model

        def predict(self, X_in):
            X_p = self.preprocessor.transform(X_in)
            if hasattr(X_p, "toarray"):
                X_p = X_p.toarray()
            return self.model.predict(X_p)

    xgb_es_pipeline = PreprocessedModel(preprocess_plain, xgb_es)
    print("XGBoost early stopping fitted.")


In [ ]:
# Fonctions d'evaluation + confusion matrix (matplotlib)

def evaluate_model(model, X_te, y_te, label):
    preds = model.predict(X_te)
    metrics = {
        "model": label,
        "accuracy": accuracy_score(y_te, preds),
        "balanced_accuracy": balanced_accuracy_score(y_te, preds),
        "f1_macro": f1_score(y_te, preds, average="macro"),
        "f1_weighted": f1_score(y_te, preds, average="weighted"),
    }

    print(f"\n[{label}] Metrics:")
    for k, v in metrics.items():
        if k != "model":
            print(f"  {k}: {v:.4f}")

    print("\nClassification report:", classification_report(y_te, preds, target_names=class_labels))

    labels = list(range(len(class_labels)))
    cm = confusion_matrix(y_te, preds, labels=labels)
    plt.figure(figsize=(6, 4))
    plt.imshow(cm, cmap="Blues")
    plt.title(f"Confusion matrix - {label}")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.xticks(ticks=range(len(labels)), labels=class_labels, rotation=45, ha="right")
    plt.yticks(ticks=range(len(labels)), labels=class_labels)
    plt.colorbar()
    plt.tight_layout()
    plt.show()

    return metrics


def plot_search_results(search, title):
    if search is None or not hasattr(search, "cv_results_"):
        return
    df = pd.DataFrame(search.cv_results_)
    score_col = "mean_test_score"
    param_cols = [c for c in df.columns if c.startswith("param_")]
    if not param_cols:
        return

    plt.figure(figsize=(7, 4))
    if len(param_cols) == 1:
        x = df[param_cols[0]].astype(str)
        y = df[score_col]
        plt.plot(x, y, marker="o")
        plt.title(title)
        plt.xlabel(param_cols[0].replace("param_", ""))
        plt.ylabel(score_col)
        plt.xticks(rotation=45, ha="right")
    elif len(param_cols) == 2:
        pivot = df.pivot_table(values=score_col, index=param_cols[0], columns=param_cols[1])
        plt.imshow(pivot.values, aspect="auto", cmap="viridis")
        plt.title(title)
        plt.xlabel(param_cols[1].replace("param_", ""))
        plt.ylabel(param_cols[0].replace("param_", ""))
        plt.colorbar(label=score_col)
        plt.xticks(ticks=range(len(pivot.columns)), labels=[str(x) for x in pivot.columns], rotation=45, ha="right")
        plt.yticks(ticks=range(len(pivot.index)), labels=[str(x) for x in pivot.index])
    else:
        top = df.sort_values(score_col, ascending=False).head(15)
        plt.bar(range(len(top)), top[score_col])
        plt.title(title + " (top 15)")
        plt.xlabel("trial")
        plt.ylabel(score_col)

    plt.tight_layout()
    plt.show()


In [ ]:
# Evaluation des modeles
results = []

results.append(evaluate_model(logreg_base, X_test, y_test, "LogReg baseline"))
results.append(evaluate_model(logreg_search.best_estimator_, X_test, y_test, "LogReg tuned"))
results.append(evaluate_model(rf_search.best_estimator_, X_test, y_test, "RandomForest tuned"))

if xgb_search is not None:
    results.append(evaluate_model(xgb_search.best_estimator_, X_test, y_test, "XGBoost tuned"))

if xgb_es_pipeline is not None:
    results.append(evaluate_model(xgb_es_pipeline, X_test, y_test, "XGBoost early_stop"))

# Visualisation des recherches hyperparametres
plot_search_results(logreg_search, "LogReg hyperparameter search")
plot_search_results(rf_search, "RandomForest hyperparameter search")
plot_search_results(xgb_search, "XGBoost hyperparameter search")


In [ ]:
# Comparaison finale + sauvegarde
results_df = pd.DataFrame(results).sort_values("f1_macro", ascending=False)
print(results_df)

# Bar chart comparison
plt.figure(figsize=(7, 4))
plt.bar(results_df["model"], results_df["f1_macro"], label="F1 macro")
plt.plot(results_df["model"], results_df["accuracy"], marker="o", color="orange", label="Accuracy")
plt.xticks(rotation=30, ha="right")
plt.title("Model comparison")
plt.ylabel("score")
plt.legend()
plt.tight_layout()
plt.show()

# Save best model
MODELS_DIR = Path("models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

best_row = results_df.iloc[0]
model_name = best_row["model"]

if model_name == "LogReg baseline":
    best_model = logreg_base
elif model_name == "LogReg tuned":
    best_model = logreg_search.best_estimator_
elif model_name == "RandomForest tuned":
    best_model = rf_search.best_estimator_
elif model_name == "XGBoost tuned":
    best_model = xgb_search.best_estimator_
else:
    best_model = logreg_search.best_estimator_

model_path = MODELS_DIR / "best_severity_model.joblib"
joblib.dump(best_model, model_path)

metrics_path = MODELS_DIR / "severity_metrics.json"
metrics_path.write_text(json.dumps(results, indent=2))

print("Saved model:", model_path)
print("Saved metrics:", metrics_path)


## Conclusion

Ce notebook entraine et compare Logistic Regression, Random Forest et XGBoost pour la classification multi-classe `severity`.
Les meilleurs hyperparametres, les scores et les visualisations sont presentes pour faciliter le choix final.
